# Lección 7.4 — Cómo priorizar qué hiperparámetros ajustar primero

> **Módulo 7 · Regularización y ajuste de hiperparámetros** · Nivel 3 — Practicante
> ⏱️ Tiempo estimado: 90–110 min (lección con ejercicios intercalados ≈ 65 min · ejercicios finales ≈ 30 min)
> 📋 Requisitos previos: Lección 5.4 (árboles), Lección 6.2 (Random Forest), Lecciones 7.1–7.3

### 🎯 Objetivos
Al terminar esta lección podrás:
- Explicar por qué **unos pocos hiperparámetros** deciden casi todo el desempeño de un modelo.
- **Medir** qué tan sensible es un modelo a cada hiperparámetro y ordenarlos por importancia.
- Usar una **tabla de prioridades** por modelo y una **receta en 5 pasos** para ajustar con presupuesto limitado.
- Saber **cuándo parar** de ajustar: cuando la mejora es más pequeña que el ruido, o cuando el problema son los datos.

### 🧭 Cómo usar este notebook
| Icono | Significa |
|---|---|
| 📖 | Lee: explicación del concepto |
| ▶️ | Ejecuta la celda y observa el resultado |
| 🔮 | **Predice** antes de ejecutar: escribe tu apuesta y luego compruébala |
| ✋ | **Tu turno**: ejercicio. Las respuestas están ocultas en "👀 Ver respuesta" |
| ✏️ | Modifica un valor en el código y vuelve a ejecutar |

> 💡 No necesitas saber programar. El código está para que **veas** los conceptos en acción; solo tendrás que cambiar números.

## ▶️ Preparación
Ejecuta esta celda una vez (`Shift + Enter`). Carga las herramientas que usaremos.

In [ ]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import randint, uniform
from sklearn.model_selection import train_test_split, cross_val_score, RandomizedSearchCV
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.datasets import make_classification

warnings.filterwarnings("ignore")
plt.rcParams["figure.figsize"] = (8, 4.5)
print("✅ Todo listo")

---
## 1. 📖 La intuición: la ducha del hotel

Llegas a un hotel y la ducha tiene **cinco** perillas: temperatura, presión, un selector de "masaje", la inclinación de la regadera y un botón de "eco".

¿Por cuál empiezas? Por la **temperatura**. Si el agua sale helada, da igual lo bien que esté la inclinación. Después, quizá, la presión. Las demás… probablemente ni las toques.

Con los hiperparámetros pasa lo mismo:
- **Unos pocos** deciden casi todo (como la temperatura).
- Otros **afinan** detalles (la presión).
- Y varios **casi no importan**, o tienen un valor "suficientemente bueno" que no hace falta buscar (el botón de eco).

Ajustar todo a la vez es como girar las cinco perillas al azar cada mañana: tardas mucho y no sabes qué funcionó. **Priorizar** es saber cuál girar primero.

### ✋ Ejercicio 1 — Calentamiento
Piensa en un árbol de decisión (Lección 5.4). Ordena estos hiperparámetros del que crees que **más** cambia el resultado al que **menos**:
- `criterion` (`gini` o `entropy`)
- `max_depth`
- `min_samples_leaf`

¿Por qué?

✍️ *Mi orden:*

<details><summary><b>👀 Ver respuesta</b></summary>

Lo más habitual: **`max_depth` ≈ `min_samples_leaf` ≫ `criterion`**. Los dos primeros deciden **cuánto crece** el árbol, es decir, si sobreajusta o subajusta. `criterion` solo cambia la forma de medir la impureza, y Gini y entropía casi siempre eligen preguntas muy parecidas. Vamos a comprobarlo con números.
</details>

---
## 2. 📖 El concepto: medir la sensibilidad

¿Cómo saber cuánto importa una perilla? Con un experimento sencillo, **uno a la vez**:
1. Deja todos los hiperparámetros en sus valores por defecto.
2. Mueve **solo uno** a lo largo de varios valores razonables y mide el puntaje con validación cruzada.
3. Anota el **rango**: la diferencia entre el mejor y el peor puntaje.
4. Repite con cada hiperparámetro y ordénalos por rango.

Un rango grande significa "esta perilla mueve la aguja". Un rango diminuto, "da igual dónde la pongas".

La función `sensibilidad` hace exactamente eso y dibuja el resultado. Usaremos datos de 3,000 clientes con 20 características y un 10 % de etiquetas con ruido.

In [ ]:
X, y = make_classification(n_samples=3_000, n_features=20, n_informative=8, n_redundant=4,
                           flip_y=0.1, class_sep=0.8, random_state=0)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, stratify=y, random_state=0)

def sensibilidad(crear_modelo, rejilla, X, y, titulo=""):
    # Mueve un hiperparámetro a la vez (los demás quedan por defecto) y mide el rango de exactitud
    filas = []
    for hp, valores in rejilla.items():
        puntajes = [cross_val_score(crear_modelo(**{hp: v}), X, y, cv=5, n_jobs=-1).mean() for v in valores]
        filas.append({"hiperparámetro": hp, "peor": min(puntajes), "mejor": max(puntajes),
                      "rango": max(puntajes) - min(puntajes),
                      "mejor valor": str(valores[int(np.argmax(puntajes))])})
    tabla = pd.DataFrame(filas).sort_values("rango").reset_index(drop=True)
    plt.barh(tabla["hiperparámetro"], tabla["rango"], color="tab:orange")
    plt.xlabel("Rango de exactitud (mejor − peor)"); plt.title(titulo); plt.show()
    return tabla.iloc[::-1].round(3)

### 🔮 Ejercicio 2 — ¿Acertaste el orden?
Ejecuta la celda con un árbol de decisión. Compara con tu orden del Ejercicio 1.

In [ ]:
rejilla_arbol = {
    "max_depth":        [1, 2, 3, 5, 8, 12, None],
    "min_samples_leaf": [1, 5, 20, 50, 100, 200],
    "max_features":     [None, 0.5, "sqrt"],
    "splitter":         ["best", "random"],
    "criterion":        ["gini", "entropy", "log_loss"],
}
crear_arbol = lambda **hp: DecisionTreeClassifier(random_state=0, **hp)
sensibilidad(crear_arbol, rejilla_arbol, X_tr, y_tr, "Árbol de decisión: ¿qué perilla mueve la aguja?")

<details><summary><b>👀 Ver respuesta</b></summary>

- **`max_depth`**: rango ≈ **0.14** (de 0.66 con profundidad 1 a 0.80 con profundidad 5–8).
- **`min_samples_leaf`**: rango ≈ **0.08**.
- `max_features` (≈ 0.013), `splitter` (≈ 0.007) y **`criterion` (≈ 0.002)**: casi nada.

Las dos perillas que controlan **cuánto crece** el árbol mueven la aguja **decenas de veces** más que `criterion`. Ajustar `criterion` con mucho cuidado sería gastar presupuesto en el botón de eco de la ducha.
</details>

### 🔮 Ejercicio 3 — Ahora un modelo moderno
Repetimos el experimento con `HistGradientBoostingClassifier` (Lección 6.4). Antes de ejecutar:
1. ¿Crees que los rangos serán **más grandes** o **más pequeños** que los del árbol?
2. ¿Cuál crees que será el hiperparámetro más importante?

> ⏱️ Esta celda tarda un poco más (≈ 1 minuto).

✍️ *Mis predicciones:*

In [ ]:
rejilla_hgb = {
    "learning_rate":     [0.01, 0.03, 0.1, 0.3, 1.0],
    "min_samples_leaf":  [1, 5, 20, 50, 100, 200],
    "max_leaf_nodes":    [4, 8, 16, 31, 64, 128],
    "l2_regularization": [0, 0.1, 1, 10, 100],
    "max_bins":          [16, 32, 64, 128, 255],
}
crear_hgb = lambda **hp: HistGradientBoostingClassifier(max_iter=200, random_state=0, **hp)
sensibilidad(crear_hgb, rejilla_hgb, X_tr, y_tr, "HistGradientBoosting: ¿qué perilla mueve la aguja?")

<details><summary><b>👀 Ver respuesta</b></summary>

1. **Mucho más pequeños**: el mayor rango ronda **0.02**, contra 0.14 del árbol. El boosting moderno tiene buenos valores por defecto y varios "frenos" incorporados, así que es **robusto**: casi cualquier valor razonable funciona bien.
2. Los más importantes son **`min_samples_leaf`** y **`learning_rate`** (≈ 0.02 cada uno), seguidos de `max_leaf_nodes` (≈ 0.014). `l2_regularization` y `max_bins` apenas mueven la aguja (≈ 0.005–0.007).

Dos lecciones:
- La importancia de cada perilla **depende del modelo** (y algo de los datos).
- Con modelos robustos, **el ajuste da mejoras pequeñas**. Si alguien te promete +10 puntos por ajustar un gradient boosting, desconfía.
</details>

### 📖 Una limitación del experimento "uno a la vez"
Mover una perilla con las demás fijas **no ve las interacciones**. Por ejemplo, en boosting, `learning_rate` y el número de árboles van de la mano (Lección 7.2): una tasa pequeña parece mala con pocos árboles y excelente con muchos. Por eso el experimento sirve para **ordenar** prioridades, no para elegir los valores finales; eso lo hace la búsqueda (Lección 7.3).

> 💡 Optuna calcula esta importancia de forma más sofisticada, a partir de todos los intentos de una búsqueda: `optuna.importance.get_param_importances(estudio)` (código para Colab en la Lección 7.3).

---
## 3. 📖 Cómo funciona por dentro: tres tipos de perillas

Después de ajustar muchos modelos, la comunidad aprendió a clasificar los hiperparámetros en tres grupos:

| Tipo | Qué son | Qué hacer | Ejemplos |
|---|---|---|---|
| 🎯 **Críticos** | Controlan la complejidad / regularización | **Ajustar siempre**, primero | `max_depth`, `C`, `alpha`, `learning_rate`, `n_neighbors`, `gamma` |
| 🔧 **Secundarios** | Afinan detalles | Ajustar si sobra presupuesto | `max_features`, `subsample`, `l2_regularization`, `weights` |
| 📌 **"Pon uno grande y olvídate"** | Más es mejor (o igual); solo cuestan tiempo | **No buscarlos**: fijarlos en un valor generoso | `n_estimators` en Random Forest; `max_iter` con early stopping |

El tercer grupo es el que más gente ajusta sin necesidad. Veámoslo.

### 🔮 Ejercicio 4 — ¿Demasiados árboles?
En un Random Forest (Lección 6.2), si pasamos de 100 a 300 árboles, ¿la exactitud en validación cruzada **sube**, **baja** (sobreajuste) o **se queda igual**?

✍️ *Mi predicción:*

In [ ]:
for n in [5, 20, 50, 100, 300]:      # ✏️ prueba 500 si tienes paciencia
    cv = cross_val_score(RandomForestClassifier(n_estimators=n, random_state=0, n_jobs=-1), X_tr, y_tr, cv=5).mean()
    print(f"n_estimators={n:>3} → exactitud CV {cv:.3f}  " + "█" * int((cv - 0.8) * 300))

<details><summary><b>👀 Ver respuesta</b></summary>

**Se queda igual** (≈ 0.880 con 100 y con 300). Con 5 árboles el bosque es pobre (≈ 0.815), mejora rápido hasta ~100 y luego **se estanca**. Nunca baja: en Random Forest, más árboles **no** causan sobreajuste, porque cada árbol es independiente y se promedian (Lección 6.2). Solo cuestan más tiempo.

Por eso `n_estimators` de Random Forest **no se busca**: se fija "suficientemente grande" (200–500) y se dedica el presupuesto a lo que sí importa.

> ⚠️ En **boosting** es distinto: los árboles se suman en cadena y demasiados **sí** sobreajustan (Lección 7.2). Ahí `n_estimators` se deja en manos de early stopping.
</details>

### 📋 Tabla de prioridades por modelo

| Modelo | 🎯 Ajusta primero | 🔧 Luego, si hay presupuesto | 📌 No lo busques: fíjalo |
|---|---|---|---|
| Regresión logística / lineal | `C` o `alpha` | `l1_ratio`, `class_weight` | `max_iter` (súbelo hasta que converja) |
| KNN | `n_neighbors` | `weights`, `metric` | — |
| SVM | `C`, `gamma` | `kernel` | — |
| Árbol de decisión | `max_depth` o `min_samples_leaf` (o `ccp_alpha`) | `max_features` | `criterion` |
| Random Forest | `max_features`, `min_samples_leaf` | `max_depth` | `n_estimators` (200–500), `criterion` |
| Gradient boosting (HGB, XGBoost, LightGBM) | `learning_rate` (+ early stopping para el número de árboles) | `max_depth`/`max_leaf_nodes`, `min_samples_leaf` | `subsample`, `colsample`, `l2_regularization`, `max_bins` (al final) |
| Red neuronal (MLP) | `learning_rate_init`, `alpha` | `hidden_layer_sizes`, `batch_size` | `max_iter` (con early stopping) |

> 📝 Es un punto de partida basado en la experiencia, no una ley. Si tus datos son raros (muy pocos, muy desbalanceados), el experimento de sensibilidad de la Sección 2 te dirá si tu caso es distinto.

---
## 4. 📖 ¿Qué pasa si sigues ajustando? Rendimientos decrecientes

Hagamos una búsqueda aleatoria de **60 intentos** sobre el árbol de decisión y dibujemos el **mejor resultado encontrado hasta cada intento** (el "récord").

### 🔮 Ejercicio 5 — La curva del récord
¿Cómo crees que será la curva: una subida constante, o una subida rápida y luego una meseta? ¿Hacia qué intento se estancará?

✍️ *Mi predicción:*

In [ ]:
busqueda = RandomizedSearchCV(DecisionTreeClassifier(random_state=0),
                              {"max_depth": randint(1, 20), "min_samples_leaf": randint(1, 200),
                               "max_features": uniform(0.1, 0.9), "criterion": ["gini", "entropy"]},
                              n_iter=60, cv=5, n_jobs=-1, random_state=0).fit(X_tr, y_tr)

puntajes = busqueda.cv_results_["mean_test_score"]
record = np.maximum.accumulate(puntajes)
plt.plot(range(1, 61), puntajes, "o", alpha=.4, label="cada intento")
plt.plot(range(1, 61), record, "r-", lw=2, label="récord hasta ese intento")
plt.xlabel("Número de intento"); plt.ylabel("Exactitud CV"); plt.legend(); plt.grid(alpha=.3); plt.show()
for n in [5, 10, 20, 40, 60]:
    print(f"Récord tras {n:>2} intentos: {record[n - 1]:.3f}")

<details><summary><b>👀 Ver respuesta</b></summary>

Subida rápida y **meseta**: tras solo 5 intentos el récord ya es ≈ 0.786. Los 35 intentos siguientes lo suben poco más de un punto (≈ 0.800 hacia el intento 40), y los últimos 20 intentos (un tercio del presupuesto) no aportan **nada**.

Esto se repite casi siempre: **los primeros intentos encuentran la región buena; el resto pule decimales**. Una búsqueda de 20–50 intentos suele bastar para la mayoría de los modelos.
</details>

### ✋ Ejercicio 6 — Lee los intentos
Los 60 intentos también sirven para ver **qué hiperparámetro importa**. La celda dibuja la exactitud de cada intento contra el valor de cada hiperparámetro. ¿En qué gráfica ves un patrón claro? ¿En cuál una nube sin forma?

In [ ]:
resultados = pd.DataFrame(busqueda.cv_results_["params"]).assign(exactitud=puntajes)
fig, axes = plt.subplots(1, 4, figsize=(16, 3.5), sharey=True)
for ax, hp in zip(axes, ["max_depth", "min_samples_leaf", "max_features", "criterion"]):
    ax.scatter(resultados[hp], resultados["exactitud"], alpha=.6)
    ax.set_xlabel(hp)
axes[0].set_ylabel("Exactitud CV"); plt.tight_layout(); plt.show()

✍️ *Tu respuesta:*

<details><summary><b>👀 Ver respuesta</b></summary>

- **`min_samples_leaf`**: el patrón más claro. Los intentos con hojas pequeñas (≤ 25) rondan 0.79 de media; con hojas enormes (> 100) caen a ≈ 0.72.
- **`max_depth`**: patrón visible en un extremo: con profundidad 1–3 la exactitud es baja (≈ 0.70 de media); por encima, la nube sube y se mezcla.
- **`max_features`**: solo los valores muy bajos (≤ 0.3) se ven claramente peores.
- **`criterion`**: ninguna diferencia clara entre `gini` y `entropy`. Es el botón de eco.

Fíjate en que aquí `min_samples_leaf` parece más importante que `max_depth`, al revés que en el experimento uno a la vez. No es una contradicción: en la búsqueda se mueven **todas** las perillas a la vez, y con hojas grandes el árbol ya no puede crecer mucho, así que la profundidad deja de importar. Las perillas **interactúan**.

Esta lectura de los resultados es la versión "a ojo" de lo que hace Optuna con su importancia de hiperparámetros, y es justo el paso 3 de la receta que verás a continuación.
</details>

---
## 5. 📖 La receta en 5 pasos (con presupuesto limitado)

```
0. LÍNEA BASE      Entrena con los valores por defecto (y un modelo "tonto", Lección 4.7). Anota media ± desviación.
1. FIJA LO FIJO    n_estimators generoso en Random Forest; max_iter alto + early stopping en boosting/redes;
                   random_state; escalado dentro de un pipeline (Lección 7.5).
2. BÚSQUEDA AMPLIA Aleatoria sobre los 2–3 hiperparámetros 🎯 críticos, en escala log, 20–40 intentos.
3. MIRA            ¿Algún mejor valor en el borde? → amplía. ¿Qué perillas no importan? → olvídalas.
                   ¿Sobra presupuesto? → añade los 🔧 secundarios.
4. REFINA          Búsqueda estrecha (o una cuadrícula pequeña) alrededor de la región buena.
5. PARA Y CONFIRMA Si la mejora es menor que el ruido (± desviación), para. Evalúa en prueba UNA vez.
```

El paso 5 es el que más se olvida. Viste en la Lección 7.3 que una mejora de +0.005 con una desviación de ±0.016 **no se distingue del azar**. Una regla práctica:

> 📏 **Si la mejora es menor que la desviación estándar entre pliegues, trátala como ruido** y quédate con la opción más simple.

### ✋ Ejercicio 7 — ¿Mejora o ruido?
Para cada caso, decide si la mejora es convincente:

| Caso | Línea base (CV) | Ajustado (CV) |
|---|---|---|
| A | 0.812 ± 0.004 | 0.838 ± 0.005 |
| B | 0.901 ± 0.020 | 0.909 ± 0.018 |
| C | 0.755 ± 0.010 | 0.760 ± 0.011 |

✍️ *Mis respuestas:*

<details><summary><b>👀 Ver respuesta</b></summary>

- **A: convincente.** Mejora +0.026, más de cinco veces la desviación (≈ 0.005).
- **B: ruido.** +0.008 con una desviación de ≈ 0.02: los pliegues varían más que la "mejora".
- **C: dudosa / ruido.** +0.005 con una desviación de ≈ 0.01. Quédate con la línea base (más simple) o junta más evidencia (por ejemplo, validación cruzada repetida con `RepeatedStratifiedKFold`).
</details>

---
## 6. 📖 Errores comunes: a veces el problema no son las perillas

### 🔮 Ejercicio 8 — Ninguna perilla arregla datos pobres
Una cadena de tiendas quiere predecir qué clientes **compran en su tienda**. Tiene la ubicación de cada cliente (`km_este`, `km_norte`). En realidad, compran los que viven **cerca** de la tienda (a menos de ~6 km), en cualquier dirección.

Probamos una regresión logística con muchos valores de `C`. Luego le añadimos **una** columna: la distancia a la tienda.

¿Qué crees que mejorará más: ajustar `C` o añadir la columna?

✍️ *Mi predicción:*

In [ ]:
rng = np.random.default_rng(0)
n = 1_500
clientes = pd.DataFrame({"km_este": rng.uniform(-10, 10, n), "km_norte": rng.uniform(-10, 10, n)})
distancia = np.hypot(clientes.km_este - 2, clientes.km_norte + 1)          # la tienda está en (2, -1)
compra = (rng.random(n) < 1 / (1 + np.exp(1.2 * (distancia - 6)))).astype(int)
print(f"Modelo tonto ('nadie compra'): {1 - compra.mean():.3f}\n")

for C in [0.001, 0.1, 10, 1000]:
    cv = cross_val_score(make_pipeline(StandardScaler(), LogisticRegression(C=C)), clientes, compra, cv=5).mean()
    print(f"Logística con C={C:<6} → exactitud CV {cv:.3f}")

con_distancia = clientes.assign(distancia_tienda=distancia)
cv = cross_val_score(make_pipeline(StandardScaler(), LogisticRegression()), con_distancia, compra, cv=5).mean()
print(f"\nLogística (C por defecto) + columna 'distancia_tienda' → exactitud CV {cv:.3f}")

<details><summary><b>👀 Ver respuesta</b></summary>

Ningún valor de `C` supera al modelo tonto (≈ 0.689): la logística traza una **recta**, y "estar cerca" es un **círculo**. Ninguna perilla convierte una recta en un círculo.

Con la columna `distancia_tienda`, la exactitud salta a ≈ **0.89**, con `C` por defecto. **Veinte puntos** por una buena característica (Lección 2.4), contra cero puntos por ajustar.

Moraleja: si el ajuste no mejora casi nada, pregúntate si el problema está en **los datos o el tipo de modelo**, no en las perillas.
</details>

| 🚨 Síntoma | Causa probable | Qué probar |
|---|---|---|
| Búsquedas enormes con mejoras diminutas | Ajustando perillas 📌 o 🔧 en lugar de las 🎯 | Experimento de sensibilidad; tabla de prioridades |
| "Ajusté `n_estimators` del Random Forest y el mejor fue el más alto" | Es una perilla de "más es mejor" | Fijarla en 200–500 y no buscarla |
| La mejora desaparece en la prueba | Mejora menor que el ruido, o búsqueda demasiado larga (optimismo, Lección 7.3) | Comparar con la desviación; menos intentos; prueba una sola vez |
| Ninguna configuración supera la línea base | Faltan características o el modelo no encaja con el problema | Ingeniería de características (Lección 2.4), otro tipo de modelo |
| Se ajustan 8 hiperparámetros con cuadrícula | Explosión combinatoria | 2–3 críticos con búsqueda aleatoria |

---
## 7. 🧠 Resumen en 5 puntos
1. **Pocos hiperparámetros deciden casi todo**: los que controlan la complejidad o la regularización.
2. Puedes **medir** la importancia moviendo una perilla a la vez y comparando rangos, o leyendo los resultados de una búsqueda.
3. Hay perillas de **"pon uno grande y olvídate"** (`n_estimators` en Random Forest, `max_iter` con early stopping): no gastes presupuesto en ellas.
4. Receta: **línea base → fija lo fijo → búsqueda aleatoria de los críticos → mira → refina → para**. Los primeros 20–40 intentos dan casi toda la ganancia.
5. Si la mejora es menor que el **ruido** de la validación cruzada, no es una mejora. Y si nada mejora, revisa los **datos**, no las perillas.

---
## 8. 🏋️ Ejercicios finales

Ya practicaste cada idea por separado. Ahora toca **combinarlas**. Estos ejercicios son más largos (≈30 min).

### 8.1 Repaso rápido (opción múltiple)

**1.** Tienes presupuesto para 20 intentos con un Random Forest. ¿Qué hiperparámetros buscas?
- a) `n_estimators` y `criterion` · b) `max_features` y `min_samples_leaf` · c) Solo `random_state` · d) Los 15 que tiene, con cuadrícula

**2.** En un gradient boosting con early stopping, ¿cuál es la perilla 🎯 número uno?
- a) `max_bins` · b) `learning_rate` · c) `n_estimators` · d) `random_state`

**3.** Tu experimento de sensibilidad da: `C` → rango 0.09; `class_weight` → 0.01; `max_iter` → 0.000. ¿Qué haces?
- a) Ajustar los tres por igual · b) Concentrarte en `C` · c) Ajustar solo `max_iter` · d) Ninguno importa

**4.** Una búsqueda pasa de 0.871 ± 0.012 (línea base) a 0.875 ± 0.013. ¿Qué concluyes?
- a) Mejora clara · b) Probablemente ruido · c) Sobreajuste seguro · d) Hay fuga de datos

**5.** Ningún valor de `C`, `penalty` o `class_weight` hace que tu logística supere al modelo tonto. ¿Qué pruebas?
- a) 500 intentos más · b) Revisar las características o probar otro tipo de modelo · c) Bajar `max_iter` · d) Cambiar la semilla

✍️ *Mis respuestas:* 1 → _ · 2 → _ · 3 → _ · 4 → _ · 5 → _

<details><summary><b>👀 Ver respuestas</b></summary>

1. **b)** Son los 🎯 de Random Forest. `n_estimators` es de "pon uno grande" y `criterion` casi no importa.
2. **b)** El número de árboles lo decide early stopping; la tasa marca cómo se recorre el camino (Lección 7.2).
3. **b)** `C` mueve la aguja 9 veces más que `class_weight`, y `max_iter` nada (siempre que el modelo converja).
4. **b)** +0.004 frente a una desviación de ≈ 0.012: no se distingue del azar.
5. **b)** Si ninguna perilla ayuda, el cuello de botella son los datos o el tipo de modelo (Ejercicio 8).
</details>

### 8.2 Preguntas para pensar (sin ejecutar)

**A.** ¿Por qué el experimento "uno a la vez" podría **subestimar** la importancia de `learning_rate` en un gradient boosting con un número **fijo** de árboles?

**B. Caso de análisis 💼** Tu equipo tiene **una semana** y un gradient boosting que ya funciona razonablemente bien en producción. Hay dos propuestas:
- **Opción A:** dedicar la semana a una búsqueda de hiperparámetros enorme.
- **Opción B:** dedicar 1 día a una búsqueda aleatoria de 40 intentos y 4 días a crear nuevas características con el equipo de negocio.

¿Cuál apoyas y por qué?

✍️ *Mis respuestas:*

<details><summary><b>👀 Ver respuestas</b></summary>

**A.** Porque `learning_rate` **interactúa** con el número de árboles. Con un número fijo (p. ej. 100), una tasa pequeña parece mala (no le da tiempo a aprender) y una grande parece buena. Con early stopping, la tasa pequeña recibe los árboles que necesita y suele salir igual o mejor. El experimento de una en una solo ve "un corte" del terreno.

**B.** En general, la **Opción B**. El modelo ya funciona y los gradient boosting son robustos: una búsqueda corta captura casi toda la ganancia posible del ajuste (rendimientos decrecientes, Sección 4). En cambio, una buena característica puede dar saltos grandes (Ejercicio 8). Una búsqueda enorme, además, infla el mejor puntaje por suerte (Lección 7.3). La Opción A solo tendría sentido si ya se agotaron las ideas de datos.
</details>

### 8.3 🧩 Práctica integradora: Random Forest con presupuesto de 10 intentos

Una empresa de software quiere predecir qué **pruebas gratuitas** se convertirán en suscripciones de pago. Tienes 2,000 registros con **40 características**, pero sospechas que pocas son útiles. Tu jefe te da un presupuesto de **10 intentos** de búsqueda para un Random Forest.

Lo harás en 5 pasos cortos. Cada uno tiene su respuesta: **no avances sin revisarla**.

**Paso 1 ▶️ — Datos y línea base**

In [ ]:
X_p, y_p = make_classification(n_samples=2_000, n_features=40, n_informative=5, n_redundant=0,
                               flip_y=0.15, random_state=2)
X_p_tr, X_p_te, y_p_tr, y_p_te = train_test_split(X_p, y_p, test_size=0.25, stratify=y_p, random_state=0)

bosque = lambda **hp: RandomForestClassifier(n_estimators=200, random_state=0, n_jobs=-1, **hp)   # 📌 fijo: 200 árboles
cv_base = cross_val_score(bosque(), X_p_tr, y_p_tr, cv=5)
print(f"Línea base: exactitud CV = {cv_base.mean():.3f} ± {cv_base.std():.3f}")

**Paso 2 🔮 — ¿Cuáles importan?** Antes de gastar el presupuesto, mide la sensibilidad de 4 hiperparámetros. Según la tabla de prioridades, ¿cuáles esperas arriba?

> ⏱️ Esta celda puede tardar 1–2 minutos.

✍️ *Mi apuesta:*

In [ ]:
rejilla_rf = {
    "max_features":     ["sqrt", 0.05, 0.2, 0.5, 1.0],
    "min_samples_leaf": [1, 3, 10, 30, 100],
    "max_depth":        [3, 6, 12, None],
    "criterion":        ["gini", "entropy"],
}
sensibilidad(bosque, rejilla_rf, X_p_tr, y_p_tr, "Random Forest: sensibilidad")

<details><summary><b>👀 Ver respuesta</b></summary>

Arriba quedan **`min_samples_leaf`** y **`max_features`** (rangos ≈ 0.03), luego `max_depth` (≈ 0.02) y, muy abajo, **`criterion`** (≈ 0.003). Coincide con la tabla de prioridades.

🕵️ Mira también la columna **"mejor valor"**: para los cuatro hiperparámetros, el mejor es… **el valor por defecto** (`min_samples_leaf=1`, `max_features="sqrt"`, `max_depth=None`, `gini`). Es una pista fuerte de que la búsqueda va a encontrar poco que mejorar.

Fíjate en `max_features = 0.05`: con 40 columnas, cada división solo mira 2 al azar, y como casi todas son ruido, muchas veces no tiene ninguna útil. Por eso hunde el resultado.
</details>

**Paso 3 ▶️ — Gasta el presupuesto con cabeza.** Dos búsquedas aleatorias de **10 intentos** cada una:
- **Enfocada:** solo los dos 🎯 (`max_features`, `min_samples_leaf`).
- **Dispersa:** los cuatro hiperparámetros a la vez.

In [ ]:
enfocada = {"max_features": uniform(0.1, 0.9), "min_samples_leaf": randint(1, 30)}
dispersa = {**enfocada, "max_depth": [3, 6, 12, None], "criterion": ["gini", "entropy"]}

busquedas = {}
for nombre, espacio in [("Enfocada", enfocada), ("Dispersa", dispersa)]:
    busquedas[nombre] = RandomizedSearchCV(bosque(), espacio, n_iter=10, cv=5, random_state=0).fit(X_p_tr, y_p_tr)
    b = busquedas[nombre]
    print(f"{nombre:<9} mejor exactitud CV = {b.best_score_:.3f} · mejores: "
          f"{ {k: (round(float(v), 2) if isinstance(v, float) else v) for k, v in b.best_params_.items()} }")

<details><summary><b>👀 Ver respuesta</b></summary>

Las dos búsquedas empatan (≈ **0.845**) y encuentran el mismo `max_features` (≈ 0.49) y `min_samples_leaf` (7): como usan la misma semilla, sortearon los mismos valores para esas dos perillas. La búsqueda dispersa, además, movió `max_depth` y `criterion`… y **no ganó nada** con ello: eran perillas que el Paso 2 ya señalaba como poco importantes. Con espacios más grandes (más hiperparámetros, más modelos), gastar intentos en perillas irrelevantes sale mucho más caro.
</details>

**Paso 4 ✋ — ¿Mejora o ruido?** Compara la mejora del ganador con la desviación de la línea base.

In [ ]:
mejor_nombre = max(busquedas, key=lambda k: busquedas[k].best_score_)
ganador = busquedas[mejor_nombre]
mejora = ganador.best_score_ - cv_base.mean()
print(f"Ganador: búsqueda {mejor_nombre}")
print(f"Mejora sobre la línea base: {mejora:+.3f}   ·   desviación de la línea base: {cv_base.std():.3f}")

✍️ *Mi conclusión:*

<details><summary><b>👀 Ver respuesta</b></summary>

La mejora es de apenas **+0.001**, contra una desviación de ≈ **0.018**: es **ruido**, sin discusión. Y eso que el `best_score_` está algo inflado por la selección (Lección 7.3). Ya lo anticipaba el Paso 2: los mejores valores eran los de por defecto.
</details>

**Paso 5 ▶️ — Examen final, una sola vez.**

In [ ]:
final = ganador.best_estimator_                       # ya reentrenado con todo el entrenamiento
base = bosque().fit(X_p_tr, y_p_tr)
print(f"Ajustado  : exactitud prueba {final.score(X_p_te, y_p_te):.3f}")
print(f"Línea base: exactitud prueba {base.score(X_p_te, y_p_te):.3f}")

<details><summary><b>👀 Ver respuesta</b></summary>

En la prueba, el modelo ajustado y la línea base **empatan** (≈ 0.850 los dos). Conclusión honesta para tu jefe: *"Con 10 intentos confirmamos que el Random Forest por defecto ya estaba cerca de su techo. Para mejorar de verdad, hay que trabajar en las características: de las 40 columnas, sospechamos que solo unas pocas aportan"*. Cómo descubrir **cuáles** es el tema de la Lección 7.6.
</details>

---
## 📚 Para profundizar
- 📄 **Artículo:** Probst, Boulesteix y Bischl (2019), *Tunability: Importance of Hyperparameters of Machine Learning Algorithms*: mide cuánto se gana ajustando cada hiperparámetro de varios modelos.
- 📘 **Guía:** Google, *Deep Learning Tuning Playbook* (GitHub): un recetario para priorizar en redes neuronales.
- 🧪 **Optuna:** `optuna.importance.get_param_importances` y `plot_param_importances`.

---
⬅️ **Anterior:** [7.3 Estrategias de búsqueda: cuadrícula, aleatoria, bayesiana y *successive halving*](7.3-estrategias-de-busqueda.ipynb) · ➡️ **Siguiente:** [7.5 Pipelines: evitar fugas mientras se ajusta](7.5-pipelines-sin-fugas.ipynb)